# pre loads

In [1]:
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import os
from tqdm import tqdm
import scipy
from scipy import signal
from scipy.signal import find_peaks
import pickle
from scipy.signal.windows import dpss
from scipy.fft import rfft, rfftfreq
from sklearn.feature_selection import mutual_info_regression

In [2]:
def picotar(x, J, passo):
    N = x.shape[0]
    Nj = (N-J)//passo + 1
    X = np.zeros((Nj, J))
    for i in range(Nj):
        X[i,:] = x[(i*passo):(i*passo+J)]
    return X

In [3]:
# Vamos carregar os dados do RealWorld
with open('C:\\Meu Drive\\Doutorado Unicamp\\Projeto\\github\\preditor-autoencoder\\Udata.pkl', 'rb') as file:
    Udata = pickle.load(file)

# Vamos remover a atividade 'jumping'
actis = ['climbingdown', 'climbingup', 'lying', 'running', 'sitting', 'standing', 'walking']
posis = ['chest', 'forearm', 'head', 'shin', 'thigh', 'upperarm', 'waist']
users = ['proband' + x for x in np.arange(1,16).astype(str)]
# proband2 não tem acc_climbingup_forearm
users.remove('proband2')
# Vamos remover usuários com menos de 21000 amostras
users.remove('proband1')
users.remove('proband4')
users.remove('proband7')
users.remove('proband14')

In [4]:
# Vamos organizar os dados em um DataFrame para facilitar a visualização
# Vamos escolher apenas os acelerômetros de um usuário correndo em todos os domínios (posições dos sensores)
# Vamos escolher apenas 10 segundos de dados (500 amostras, do ponto 3500 até 4000) para facilitar a visualização
nu = 3  
na = 3  # Atividade 3, que é running
fs = 50
df_RW = pd.DataFrame({
    'time': np.arange(500) / fs,
    'chest_X': np.array(Udata[nu][0])[na, 3500:4000, 0],
    'chest_Y': np.array(Udata[nu][0])[na, 3500:4000, 1],
    'chest_Z': np.array(Udata[nu][0])[na, 3500:4000, 2],
    'forearm_X': np.array(Udata[nu][1])[na, 3500:4000, 0],
    'forearm_Y': np.array(Udata[nu][1])[na, 3500:4000, 1],
    'forearm_Z': np.array(Udata[nu][1])[na, 3500:4000, 2],
    'head_X': np.array(Udata[nu][2])[na, 3500:4000, 0],
    'head_Y': np.array(Udata[nu][2])[na, 3500:4000, 1],
    'head_Z': np.array(Udata[nu][2])[na, 3500:4000, 2],
    'shin_X': np.array(Udata[nu][3])[na, 3500:4000, 0],
    'shin_Y': np.array(Udata[nu][3])[na, 3500:4000, 1],
    'shin_Z': np.array(Udata[nu][3])[na, 3500:4000, 2],
    'thigh_X': np.array(Udata[nu][4])[na, 3500:4000, 0],
    'thigh_Y': np.array(Udata[nu][4])[na, 3500:4000, 1],
    'thigh_Z': np.array(Udata[nu][4])[na, 3500:4000, 2],
    'upperarm_X': np.array(Udata[nu][5])[na, 3500:4000, 0],
    'upperarm_Y': np.array(Udata[nu][5])[na, 3500:4000, 1],
    'upperarm_Z': np.array(Udata[nu][5])[na, 3500:4000, 2],
    'waist_X': np.array(Udata[nu][6])[na, 3500:4000, 0],
    'waist_Y': np.array(Udata[nu][6])[na, 3500:4000, 1],
    'waist_Z': np.array(Udata[nu][6])[na, 3500:4000, 2]
})
# Vamos filtrar os sinais com um filtro Butterworth passa-altas com fc = 0.3 Hz
for col in df_RW.columns:
    if col != 'time':
        b, a = signal.butter(5, 0.3, 'hp', fs=fs)
        zi = signal.lfilter_zi(b, a)
        z, _ = signal.lfilter(b, a, df_RW[col], zi=zi*df_RW[col][0])
        df_RW[col] = z
# Vamos normalizar os sinais pela energia dos três eixos combinados
ener = (df_RW.values[:, 1:]**2).sum(axis=0).reshape(-1, 3).sum(axis=1)
for i in range(0, df_RW.shape[1]-1, 3):
    df_RW.iloc[:, i+1:i+4] = df_RW.iloc[:, i+1:i+4] / np.sqrt(ener[i//3])

In [5]:
filepath = 'Kinematics_BodyKinematics_acc_global.sto'
df_acc_global = pd.read_csv(filepath, sep='\s+', skiprows=18)

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\dami_\AppData\Local\Temp\ipykernel_6308\1274878355.py:2: SyntaxWarning: invalid escape sequence '\s'
  df_acc_global = pd.read_csv(filepath, sep='\s+', skiprows=18)


In [6]:
# Agora vamos organizar os dados do arquivo .sto no mesmo formato do DataFrame do RealWorld
# Mas antes temos de associar os nomes dos sensores do arquivo .sto com as posições dos sensores do RealWorld
# chest -> torso, forearm -> hand_l, head -> torso, shin -> tibia_l, thigh -> femur_l, upperarm -> humerus_l, waist -> pelvis
# Por fim, vamos reamostrar os dados do arquivo .sto para 50 Hz, que é a frequência de amostragem do RealWorld, usando scipy.signal.resample
df_sto = pd.DataFrame({
    'time': np.arange(500) / fs,
    'chest_X': scipy.signal.resample(df_acc_global['torso_X'], 500),
    'chest_Y': scipy.signal.resample(df_acc_global['torso_Y'], 500),
    'chest_Z': scipy.signal.resample(df_acc_global['torso_Z'], 500),
    'forearm_X': scipy.signal.resample(df_acc_global['hand_l_X'], 500),
    'forearm_Y': scipy.signal.resample(df_acc_global['hand_l_Y'], 500),
    'forearm_Z': scipy.signal.resample(df_acc_global['hand_l_Z'], 500),
    'head_X': scipy.signal.resample(df_acc_global['torso_X'], 500),
    'head_Y': scipy.signal.resample(df_acc_global['torso_Y'], 500),
    'head_Z': scipy.signal.resample(df_acc_global['torso_Z'], 500),
    'shin_X': scipy.signal.resample(df_acc_global['tibia_l_X'], 500),
    'shin_Y': scipy.signal.resample(df_acc_global['tibia_l_Y'], 500),
    'shin_Z': scipy.signal.resample(df_acc_global['tibia_l_Z'], 500),
    'thigh_X': scipy.signal.resample(df_acc_global['femur_l_X'], 500),
    'thigh_Y': scipy.signal.resample(df_acc_global['femur_l_Y'], 500),
    'thigh_Z': scipy.signal.resample(df_acc_global['femur_l_Z'], 500),
    'upperarm_X': scipy.signal.resample(df_acc_global['humerus_l_X'], 500),
    'upperarm_Y': scipy.signal.resample(df_acc_global['humerus_l_Y'], 500),
    'upperarm_Z': scipy.signal.resample(df_acc_global['humerus_l_Z'], 500),
    'waist_X': scipy.signal.resample(df_acc_global['pelvis_X'], 500),
    'waist_Y': scipy.signal.resample(df_acc_global['pelvis_Y'], 500),
    'waist_Z': scipy.signal.resample(df_acc_global['pelvis_Z'], 500)
})
# Vamos filtrar os sinais com um filtro Butterworth passa-altas com fc = 0.3 Hz
for col in df_sto.columns:
    if col != 'time':
        b, a = signal.butter(5, 0.3, 'hp', fs=fs)
        zi = signal.lfilter_zi(b, a)
        z, _ = signal.lfilter(b, a, df_sto[col], zi=zi*df_sto[col][0])
        df_sto[col] = z
# Vamos normalizar os sinais pela energia dos três eixos combinados
ener = (df_sto.values[:, 1:]**2).sum(axis=0).reshape(-1, 3).sum(axis=1)
for i in range(0, df_sto.shape[1]-1, 3):
    df_sto.iloc[:, i+1:i+4] = df_sto.iloc[:, i+1:i+4] / np.sqrt(ener[i//3])

In [7]:
file_path = 'subject02_running.trc'

# 1. Ler apenas os nomes dos marcadores (Linha 4)
# O cabeçalho real de nomes está na 4ª linha (index 3)
marker_header = pd.read_csv(file_path, sep='\t', skiprows=3, nrows=0).columns.tolist()

# 2. Criar nomes de colunas amigáveis (Ex: R.ASIS_X, R.ASIS_Y, R.ASIS_Z)
final_cols = ['Frame', 'Time']
# Os marcadores começam da 3ª coluna no arquivo .trc
markers = [m for m in marker_header if m not in ['Frame#', 'Time'] and not m.startswith('Unnamed')]

for marker in markers:
    final_cols.extend([f"{marker}_X", f"{marker}_Y", f"{marker}_Z"])
final_cols.append('Err')

# 3. Ler os dados (começando da Linha 6)
# Usamos skiprows=5 para pular as 5 linhas de metadados e cabeçalho
df = pd.read_csv(file_path, sep='\t', skiprows=5, names=final_cols[1:])

# Remover colunas extras que podem aparecer por causa de abas sobrando no final do arquivo
df = df.dropna(axis=1, how='all')
t = df['Time'].values
dt = t[1] - t[0]

In [8]:
# Agora vamos organizar os dados dos marcadores do arquivo .trc no mesmo formato do DataFrame do RealWorld
# Mas antes temos de associar os nomes dos marcadores do arquivo .trc com as posições dos sensores do RealWorld
# chest -> Sternum, forearm -> L.Wrist.Med, head -> L.Acromium, shin -> L.Midfoot.Lat, thigh -> L.Thigh.Upper, upperarm -> L.Bicep, waist -> L.ASIS
# Vamos lembrar de derivar os dados de posição dos marcadores para obter as acelerações e depois reamostrar para 50 Hz
# Vamos criar um mapeamento dos marcadores para as posições dos sensores do RealWorld para todos os eixos (como chest_X -> Sternum_X etc.)
marker_to_sensor = {
    'chest_X': 'Sternum_X',
    'chest_Y': 'Sternum_Y',
    'chest_Z': 'Sternum_Z',
    'forearm_X': 'L.Wrist.Med_X',
    'forearm_Y': 'L.Wrist.Med_Y',
    'forearm_Z': 'L.Wrist.Med_Z',
    'head_X': 'L.Acromium_X',
    'head_Y': 'L.Acromium_Y',
    'head_Z': 'L.Acromium_Z',
    'shin_X': 'L.Midfoot.Lat_X',
    'shin_Y': 'L.Midfoot.Lat_Y',
    'shin_Z': 'L.Midfoot.Lat_Z',
    'thigh_X': 'L.Thigh.Upper_X',
    'thigh_Y': 'L.Thigh.Upper_Y',
    'thigh_Z': 'L.Thigh.Upper_Z',
    'upperarm_X': 'R.Bicep_X',  # O L.Bicep está com todos os valores iguais a zero, então vamos usar o R.Bicep para representar o upperarm
    'upperarm_Y': 'R.Bicep_Y',
    'upperarm_Z': 'R.Bicep_Z',
    'waist_X': 'L.ASIS_X',
    'waist_Y': 'L.ASIS_Y',
    'waist_Z': 'L.ASIS_Z'
}
df_trc = pd.DataFrame({
    'time': np.arange(500) / fs
})
for sensor, marker in marker_to_sensor.items():
    # Derivar os dados de posição para obter a aceleração
    pos = df[marker].values
    vel = np.gradient(pos, dt)
    acc = np.gradient(vel, dt)
    # Reamostrar para 50 Hz
    df_trc[sensor] = scipy.signal.resample(acc, 500)
# Vamos filtrar os sinais com um filtro Butterworth passa-altas com fc = 0.3 Hz
for col in df_trc.columns:
    if col != 'time':
        b, a = signal.butter(5, 0.3, 'hp', fs=fs)
        zi = signal.lfilter_zi(b, a)
        z, _ = signal.lfilter(b, a, df_trc[col], zi=zi*df_trc[col][0])
        df_trc[col] = z
# Vamos normalizar os sinais pela energia dos três eixos combinados
ener = (df_trc.values[:, 1:]**2).sum(axis=0).reshape(-1, 3).sum(axis=1)
for i in range(0, df_trc.shape[1]-1, 3):
    df_trc.iloc[:, i+1:i+4] = df_trc.iloc[:, i+1:i+4] / np.sqrt(ener[i//3])

# Estimando o mesmo f0 entre domínios (trc mocap)

In [9]:
J = 150
L = 8
f0s = np.arange(1.2, 1.7, 0.001)
F0 = []
Feats = []
d = posis[0]
x, y, z = df_trc[d+'_X'], df_trc[d+'_Y'], df_trc[d+'_Z']
X = picotar(x, J, 1)
Y = picotar(y, J, 1)
Z = picotar(z, J, 1)
D1 = np.stack((X, Y, Z), axis=2)
d = posis[1]
x, y, z = df_trc[d+'_X'], df_trc[d+'_Y'], df_trc[d+'_Z']
X = picotar(x, J, 1)
Y = picotar(y, J, 1)
Z = picotar(z, J, 1)
D2 = np.stack((X, Y, Z), axis=2)
D = np.concatenate((D1, D2), axis=2)
f0est = []
feats = []
for i in tqdm(range(D.shape[0])):
    janela = D[i,:,:]
    erro = []
    for f0 in f0s:
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.sum((janela - recon)**2)))
    f0 = f0s[np.argmin(erro)]
    f0est.append(f0)
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(janela)
    feats.append(coefs)
F0 = np.array(f0est)
Feats = np.array(feats)

100%|██████████| 351/351 [00:33<00:00, 10.37it/s]


In [17]:
# A variação natural da passada é de 3%. Para um f0 de 1.45 Hz, a variação natural seria de ~0.05 Hz.
# Isto é, o f0 poderia variar entre 1.4 Hz e 1.5 Hz sem que isso fosse considerado uma mudança de passo.
px.line(F0)

In [10]:
desc1 = []
for i in range(Feats.shape[0]):
    coefs = Feats[i,:,:3]
    aux = []
    for l in range(L):
        A = coefs[l,:]
        B = coefs[l+L,:]
        M = np.vstack((A, B)).T
        G = M.T.dot(M)
        w, v = np.linalg.eig(G)
        w = np.sort(w)[::-1]
        sige = np.sqrt(w)
        El = np.sqrt(w.sum())
        e = np.sqrt(1 - (w[1]/w[0]))
        aux.append([El, e, sige[0], sige[1]])
    desc1.append(aux)
desc1 = np.array(desc1)
desc2 = []
for i in range(Feats.shape[0]):
    coefs = Feats[i,:,3:]
    aux = []
    for l in range(L):
        A = coefs[l,:]
        B = coefs[l+L,:]
        M = np.vstack((A, B)).T
        G = M.T.dot(M)
        w, v = np.linalg.eig(G)
        w = np.sort(w)[::-1]
        sige = np.sqrt(w)
        El = np.sqrt(w.sum())
        e = np.sqrt(1 - (w[1]/w[0]))
        aux.append([El, e, sige[0], sige[1]])
    desc2.append(aux)
desc2 = np.array(desc2)
desc1.shape, desc2.shape

((351, 8, 4), (351, 8, 4))

In [12]:
fig = go.Figure()
for l in range(L):
    fig.add_trace(go.Scatter(x=desc1[:,l,0], y=desc2[:,l,0], mode='markers+lines', name=f'Harmônico {l+1}'))
fig.update_layout(title='Energia dos Harmônicos do head vs forearm', xaxis_title='Energia do head', yaxis_title='Energia do forearm', width=700, height=500)
fig.show()

In [24]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=F0, y=desc1[:,1,0], mode='markers', name=posis[d]))
fig.add_trace(go.Scatter(x=F0, y=desc2[:,1,0], mode='markers', name=posis[d]))
fig.update_layout(width=700, height=500, title='Energia do segundo harmônico ao longo do tempo', xaxis_title='F0 estimado (Hz)', yaxis_title='Energia do segundo harmônico')

In [ ]:
# vamos salvar os dados desc1[100:120,:,0] em um arquivo csv
# pd.DataFrame(desc1[100:120,:,0], columns=[f'Harmônico {l+1}' for l in range(L)]).to_csv('d1_harmonicos.csv', index=False)
# pd.DataFrame(desc2[100:120,:,0], columns=[f'Harmônico {l+1}' for l in range(L)]).to_csv('d2_harmonicos.csv', index=False)

In [21]:
for l in range(L):
    s1 = np.sign(desc1[1:,l,0] - desc1[:-1,l,0])
    s2 = np.sign(desc2[1:,l,0] - desc2[:-1,l,0])
    tC = np.zeros((2,2), dtype=int)
    tC[0,0] = np.sum(s1==1)
    tC[0,1] = np.sum(s1==-1)
    tC[1,0] = np.sum(s2==1)
    tC[1,1] = np.sum(s2==-1)
    tE = np.zeros((2,2))
    for i in range(2):
        for j in range(2):
            tE[i,j] = tC[i,:].sum() * tC[:,j].sum() / tC.sum()
    qui2 = ((tC - tE)**2 / tE).sum()
    pValor = 1 - scipy.stats.chi2.cdf(qui2, df=1)
    print(f'Qui-quadrado: {qui2:.2f}, p-valor: {pValor:.4f}')

Qui-quadrado: 0.02, p-valor: 0.8798
Qui-quadrado: 1.13, p-valor: 0.2884
Qui-quadrado: 0.46, p-valor: 0.4963
Qui-quadrado: 0.14, p-valor: 0.7054
Qui-quadrado: 0.02, p-valor: 0.8798
Qui-quadrado: 0.05, p-valor: 0.8206
Qui-quadrado: 1.12, p-valor: 0.2898
Qui-quadrado: 0.02, p-valor: 0.8798


In [30]:
J = 150
L = 8
f0s = np.arange(1.2, 1.7, 0.001)
F0 = []
Feats = []
D = []
for d in posis:
    x, y, z = df_trc[d+'_X'], df_trc[d+'_Y'], df_trc[d+'_Z']
    X = picotar(x, J, 1)
    Y = picotar(y, J, 1)
    Z = picotar(z, J, 1)
    D.append(np.stack((X, Y, Z), axis=2))
D = np.concatenate(D, axis=2)
f0est = []
feats = []
for i in tqdm(range(D.shape[0])):
    janela = D[i,:,:]
    erro = []
    for f0 in f0s:
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.sum((janela - recon)**2)))
    f0 = f0s[np.argmin(erro)]
    f0est.append(f0)
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(janela)
    feats.append(coefs)
F0 = np.array(f0est)
Feats = np.array(feats)

100%|██████████| 351/351 [00:34<00:00, 10.09it/s]


In [36]:
Desc = []
for d in range(len(posis)):
    desc = []
    for i in range(Feats.shape[0]):
        coefs = Feats[i,:,d*3:(d+1)*3]
        aux = []
        for l in range(L):
            A = coefs[l,:]
            B = coefs[l+L,:]
            M = np.vstack((A, B)).T
            G = M.T.dot(M)
            w, v = np.linalg.eig(G)
            w = np.sort(w)[::-1]
            sige = np.sqrt(w)
            El = np.sqrt(w.sum())
            e = np.sqrt(1 - (w[1]/w[0]))
            aux.append([El, e, sige[0], sige[1]])
        desc.append(aux)
    Desc.append(np.array(desc))
Desc = np.array(Desc)

In [48]:
Qui, Pv, rotulo = [], [], []
for d1 in range(len(posis)):
    for d2 in range(d1+1, len(posis)):
        for l in range(L):
            s1 = np.sign(Desc[d1,1:,l,0] - Desc[d1,:-1,l,0])
            s2 = np.sign(Desc[d2,1:,l,0] - Desc[d2,:-1,l,0])
            tC = np.zeros((2,2), dtype=int)
            tC[0,0] = np.sum(s1==1)
            tC[0,1] = np.sum(s1==-1)
            tC[1,0] = np.sum(s2==1)
            tC[1,1] = np.sum(s2==-1)
            tE = np.zeros((2,2))
            for i in range(2):
                for j in range(2):
                    tE[i,j] = tC[i,:].sum() * tC[:,j].sum() / tC.sum()
            qui2 = ((tC - tE)**2 / tE).sum()
            pValor = 1 - scipy.stats.chi2.cdf(qui2, df=1)
            Qui.append(qui2)
            Pv.append(pValor)
            rotulo.append(f'{posis[d1]} com {posis[d2]} - Harmônico {l+1}')

In [50]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=Qui, mode='markers', name='Qui-quadrado'))
fig.add_trace(go.Scatter(y=Pv, mode='markers', name='p-valor'))
fig.update_layout(title='Teste de Qui-quadrado para comparação entre domínios', xaxis_title='Par de domínios comparados', yaxis_title='Valor do teste', height=700)
fig.update_xaxes(tickmode='array', tickvals=np.arange(len(rotulo)), ticktext=rotulo)
fig.show()

# Vamos dar uma olhada nas atividades de subir e descer escadas (RW)

## Leitura dos dados

In [87]:
nu = 4
na = 0 # climbingdown
nd = 1
px.line(np.array(Udata[nu][nd])[na, :, :3])

In [ ]:
nu = 4
na = 0  # Atividade 0, que é climbingdown
fs = 50
ini, fim, dur = 5000, 6000, fim-ini
df_RW = pd.DataFrame({
    'time': np.arange(dur) / fs,
    'chest_X': np.array(Udata[nu][0])[na, ini:fim, 0],
    'chest_Y': np.array(Udata[nu][0])[na, ini:fim, 1],
    'chest_Z': np.array(Udata[nu][0])[na, ini:fim, 2],
    'forearm_X': np.array(Udata[nu][1])[na, ini:fim, 0],
    'forearm_Y': np.array(Udata[nu][1])[na, ini:fim, 1],
    'forearm_Z': np.array(Udata[nu][1])[na, ini:fim, 2],
    'head_X': np.array(Udata[nu][2])[na, ini:fim, 0],
    'head_Y': np.array(Udata[nu][2])[na, ini:fim, 1],
    'head_Z': np.array(Udata[nu][2])[na, ini:fim, 2],
    'shin_X': np.array(Udata[nu][3])[na, ini:fim, 0],
    'shin_Y': np.array(Udata[nu][3])[na, ini:fim, 1],
    'shin_Z': np.array(Udata[nu][3])[na, ini:fim, 2],
    'thigh_X': np.array(Udata[nu][4])[na, ini:fim, 0],
    'thigh_Y': np.array(Udata[nu][4])[na, ini:fim, 1],
    'thigh_Z': np.array(Udata[nu][4])[na, ini:fim, 2],
    'upperarm_X': np.array(Udata[nu][5])[na, ini:fim, 0],
    'upperarm_Y': np.array(Udata[nu][5])[na, ini:fim, 1],
    'upperarm_Z': np.array(Udata[nu][5])[na, ini:fim, 2],
    'waist_X': np.array(Udata[nu][6])[na, ini:fim, 0],
    'waist_Y': np.array(Udata[nu][6])[na, ini:fim, 1],
    'waist_Z': np.array(Udata[nu][6])[na, ini:fim, 2]
})
# Vamos filtrar os sinais com um filtro Butterworth passa-altas com fc = 0.3 Hz
for col in df_RW.columns:
    if col != 'time':
        b, a = signal.butter(5, 0.3, 'hp', fs=fs)
        zi = signal.lfilter_zi(b, a)
        z, _ = signal.lfilter(b, a, df_RW[col], zi=zi*df_RW[col][0])
        df_RW[col] = z
# Vamos normalizar os sinais pela energia dos três eixos combinados
# ener = (df_RW.values[:, 1:]**2).sum(axis=0).reshape(-1, 3).sum(axis=1)
# for i in range(0, df_RW.shape[1]-1, 3):
#     df_RW.iloc[:, i+1:i+4] = df_RW.iloc[:, i+1:i+4] / np.sqrt(ener[i//3])

## Análise f0

In [94]:
J = 150
L = 8
f0s = np.arange(0.8, 1.2, 0.001)
F0 = []
Feats = []
D = []
for d in posis:
    x, y, z = df_RW[d+'_X'], df_RW[d+'_Y'], df_RW[d+'_Z']
    X = picotar(x, J, 1)
    Y = picotar(y, J, 1)
    Z = picotar(z, J, 1)
    D.append(np.stack((X, Y, Z), axis=2))
D = np.concatenate(D, axis=2)
f0est = []
feats = []
for i in tqdm(range(D.shape[0])):
    janela = D[i,:,:]
    erro = []
    for f0 in f0s:
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.sum((janela - recon)**2)))
    f0 = f0s[np.argmin(erro)]
    f0est.append(f0)
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(janela)
    feats.append(coefs)
F0 = np.array(f0est)
Feats = np.array(feats)

100%|██████████| 851/851 [02:46<00:00,  5.10it/s]


In [96]:
px.line(F0)
# Observe que nessa janela entre 5000 e 6000 há momentos de variações rápidas de f0 (nos instantes 69, 241 e 505)
# Isso é consistente com o que é observado no sinal bruto, isto é, variações def0 na faixa de +-0,1Hz
# Ainda não é claro a razão dessas variações rápidas nessa atividade de descida de escada...

In [97]:
Desc = []
for d in range(len(posis)):
    desc = []
    for i in range(Feats.shape[0]):
        coefs = Feats[i,:,d*3:(d+1)*3]
        aux = []
        for l in range(L):
            A = coefs[l,:]
            B = coefs[l+L,:]
            M = np.vstack((A, B)).T
            G = M.T.dot(M)
            w, v = np.linalg.eig(G)
            w = np.sort(w)[::-1]
            sige = np.sqrt(w)
            El = np.sqrt(w.sum())
            e = np.sqrt(1 - (w[1]/w[0]))
            aux.append([El, e, sige[0], sige[1]])
        desc.append(aux)
    Desc.append(np.array(desc))
Desc = np.array(Desc)

In [98]:
Qui, Pv, rotulo = [], [], []
for d1 in range(len(posis)):
    for d2 in range(d1+1, len(posis)):
        for l in range(L):
            s1 = np.sign(Desc[d1,1:,l,0] - Desc[d1,:-1,l,0])
            s2 = np.sign(Desc[d2,1:,l,0] - Desc[d2,:-1,l,0])
            tC = np.zeros((2,2), dtype=int)
            tC[0,0] = np.sum(s1==1)
            tC[0,1] = np.sum(s1==-1)
            tC[1,0] = np.sum(s2==1)
            tC[1,1] = np.sum(s2==-1)
            tE = np.zeros((2,2))
            for i in range(2):
                for j in range(2):
                    tE[i,j] = tC[i,:].sum() * tC[:,j].sum() / tC.sum()
            qui2 = ((tC - tE)**2 / tE).sum()
            pValor = 1 - scipy.stats.chi2.cdf(qui2, df=1)
            Qui.append(qui2)
            Pv.append(pValor)
            rotulo.append(f'{posis[d1]} com {posis[d2]} - Harmônico {l+1}')

In [99]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=Qui, mode='markers', name='Qui-quadrado'))
fig.add_trace(go.Scatter(y=Pv, mode='markers', name='p-valor'))
fig.update_layout(title='Teste de Qui-quadrado para comparação entre domínios', xaxis_title='Par de domínios comparados', yaxis_title='Valor do teste', height=700)
fig.update_xaxes(tickmode='array', tickvals=np.arange(len(rotulo)), ticktext=rotulo)
fig.show()